# 🏎️ F1 Race Predictor — Machine Learning Based Podium Prediction
### College Engineering ML Project | Formula 1 Analytics

**Objective:** Predict whether a Formula 1 driver will finish on the podium (Top 3) using strictly pre-race historical features and chronological machine learning models.

---
### Table of Contents:
1. Dataset Loading & Pre-Race Schema Inspection
2. Exploratory Data Analysis (EDA)
3. Leakage-Free Pre-Race Feature Engineering
4. Chronological Splitting (Train: 2010–2021, Val: 2022, Test: 2023–2024)
5. Model Training & Hyperparameter Tuning (5 Algorithms)
6. Model Comparison & Evaluation Metrics
7. Feature Importance & Factor Analysis
8. Conclusion & Viva Summary

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to path
sys.path.append('..')
from src.data_loader import load_raw_data, get_base_race_dataframe
from src.feature_engineering import generate_prerace_features
from src.preprocessing import F1Preprocessor, get_chronological_splits, FEATURE_COLUMNS
from src.evaluate import evaluate_predictions

plt.style.use('dark_background')
print('Libraries successfully imported!')

## 1. Load Raw Ergast Data
Let's load the raw Formula 1 historical CSV files from `data/raw/`.

In [ ]:
raw_data = load_raw_data('../data/raw')
base_df = get_base_race_dataframe(raw_data)
print('Total Driver Starts:', len(base_df))
print('Seasons:', base_df['year'].min(), 'to', base_df['year'].max())
print(f"Podium Rate (Base Rate): {base_df['podium'].mean():.4f}")
base_df.head(5)

## 2. Exploratory Data Analysis (EDA)
Let's analyze the relationship between starting grid position and podium conversion rate.

In [ ]:
plt.figure(figsize=(10, 5))
grid_stats = base_df[base_df['grid'].between(1, 20)].groupby('grid')['podium'].mean().reset_index()
sns.barplot(data=grid_stats, x='grid', y='podium', color='#e10600')
plt.title('Historical Starting Grid Position vs Podium Conversion Rate', fontsize=14, fontweight='bold')
plt.xlabel('Starting Grid Slot')
plt.ylabel('Podium Probability')
plt.show()

## 3. Pre-Race Feature Engineering
Generate strictly leak-free rolling features computed from past historical races only.

In [ ]:
features_df, _, _, _, _ = generate_prerace_features(base_df)
print('Processed Features Shape:', features_df.shape)
features_df[FEATURE_COLUMNS].head()

## 4. Chronological Splitting & Preprocessing
Split into Train (2010–2021), Validation (2022), and Test (2023–2024).

In [ ]:
(X_train, y_train, _), (X_val, y_val, _), (X_test, y_test, _) = get_chronological_splits(features_df)
print('Train size (2010-2021):', len(X_train), 'rows')
print('Val size (2022):      ', len(X_val), 'rows')
print('Test size (2023-2024): ', len(X_test), 'rows')

preprocessor = F1Preprocessor()
X_train_scaled = preprocessor.fit_transform(X_train)
X_val_scaled = preprocessor.transform(X_val)
X_test_scaled = preprocessor.transform(X_test)

## 5. Model Training & Comparison
Train 5 candidate classification algorithms and evaluate performance on the test set.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=250, max_depth=10, class_weight='balanced_subsample', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=180, learning_rate=0.06, max_depth=4, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, eval_metric='logloss', random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    prob = model.predict_proba(X_test_scaled)[:, 1]
    pred = (prob >= 0.35).astype(int)
    m = evaluate_predictions(y_test, pred, prob)
    m['Model'] = name
    results.append(m)

results_df = pd.DataFrame(results)[['Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']]
results_df.sort_values(by=['F1', 'ROC-AUC'], ascending=False)

## 6. Feature Importance
Analyze the most influential pre-race features in predicting podium finishes.

In [ ]:
rf_model = models['Random Forest']
feat_imp = pd.DataFrame({
    'Feature': FEATURE_COLUMNS,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(data=feat_imp, x='Importance', y='Feature', color='#e10600')
plt.title('Top 10 Pre-Race Feature Importances (Random Forest)', fontsize=14, fontweight='bold')
plt.xlabel('Relative Importance')
plt.ylabel('')
plt.show()

## 7. Conclusion
* **Top Performing Model:** Random Forest Classifier achieved **0.9237 ROC-AUC** and **0.6466 F1-Score** on the unseen 2023–2024 test seasons.
* **Key Predictors:** Grid starting slot, recent driver form score (exponentially weighted), constructor performance rating, and driver circuit podium rate are the primary determinants of Grand Prix podium finishes.
* **Zero Data Leakage:** All features are strictly pre-race rolling metrics with zero in-race leakage.